# DTW 클러스터별 iTransformer 어텐션 패턴 분석

**목적**: 학습된 iTransformer의 variate-attention을 DTW 클러스터별로 비교하여
각 클러스터에서 지배적인 기술적 지표 관계를 해석합니다.

| 단계 | 내용 |
|------|------|
| 1 | 클러스터별 대표 종목 시계열 추출 (parquet) |
| 2 | instance normalization → iTransformer 추론 |
| 3 | 어텐션 맵 평균 → 클러스터 간 히트맵 비교 |
| 4 | 피처 중요도 순위 및 클러스터 특성 해석 |

> **전제**: `backbone_comparison_colab.ipynb`를 먼저 실행하여 `itransformer_best.pt`가 Drive에 저장되어 있어야 합니다.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q h5py pyarrow tqdm

In [ ]:
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
import torch.nn as nn
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['axes.unicode_minus'] = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

DRIVE_ROOT = Path('/content/drive/MyDrive/grad_project')
DATA_DIR   = DRIVE_ROOT / 'data'
MODEL_DIR  = DRIVE_ROOT / 'models'
RESULT_DIR = DATA_DIR   / 'cluster_attn'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

PRICE_COLS = ['Adj_Close', 'Open', 'High', 'Low', 'Volume']
TECH_COLS  = [
    'SMA_5', 'SMA_20', 'SMA_60',
    'EMA_12', 'EMA_26',
    'MACD', 'MACD_signal', 'MACD_hist',
    'RSI_14', 'BB_width', 'Volume_ratio',
    'Return_1d', 'Return_5d',
    'Volatility_20d', 'ATR_14',
]
FEATURE_COLS = PRICE_COLS + TECH_COLS
N_FEATURES   = len(FEATURE_COLS)
SEQ_LEN      = 60

CLUSTER_LABELS = {
    0: 'C0 부동산/금융 (108)',
    1: 'C1 중공업/철강 (112)',
    2: 'C2 자동차부품/제약 (118)',
    3: 'C3 혼합 대형 (299)',
    4: 'C4 금융/화학/보험 (120)',
}
CLUSTER_COLORS = ['steelblue', 'tomato', 'darkorange', 'mediumseagreen', 'mediumpurple']

assert (MODEL_DIR / 'itransformer_best.pt').exists(), \
    'itransformer_best.pt 없음 — backbone_comparison_colab.ipynb 먼저 실행'
print('설정 완료')
print(f'피처: {N_FEATURES}개 | 입력: {SEQ_LEN}일')

## 2. 클러스터 현황

In [ ]:
cluster_df = pd.read_csv(DATA_DIR / 'cluster_assignments.csv')

print('클러스터별 종목 수:')
print(cluster_df['cluster'].value_counts().sort_index().to_string())
print()

for c in sorted(cluster_df['cluster'].unique()):
    sub  = cluster_df[cluster_df['cluster'] == c]
    top3 = sub['업종'].value_counts().head(3).index.tolist()
    print(f'{CLUSTER_LABELS[c]}: {top3}')

fig, ax = plt.subplots(figsize=(8, 3))
counts = cluster_df['cluster'].value_counts().sort_index()
bars = ax.bar([CLUSTER_LABELS[i] for i in counts.index], counts.values,
              color=CLUSTER_COLORS, alpha=0.85, edgecolor='k', lw=0.5)
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(v), ha='center', fontsize=10)
ax.set_ylabel('종목 수')
ax.set_title('DTW K-Means 클러스터 분포 (K=5, 757종목)')
ax.tick_params(axis='x', rotation=10)
plt.tight_layout()
plt.show()

## 3. iTransformer 모델 로드

In [ ]:
class iTransformer(nn.Module):
    def __init__(self, seq_len=60, n_features=20, d_model=128,
                 nhead=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.variate_embed = nn.Linear(seq_len, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model, nhead, d_model * 4, dropout,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(n_features * d_model, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x_patch, x_flat):
        x = x_flat.permute(0, 2, 1)
        x = self.variate_embed(x)
        x = self.norm(self.transformer(x))
        B, N, D = x.shape
        return self.head(x.reshape(B, N * D)).squeeze(-1)

    @torch.no_grad()
    def get_attn_maps(self, x_flat):
        self.eval()
        x = x_flat.permute(0, 2, 1)
        x = self.variate_embed(x)
        maps = []
        for layer in self.transformer.layers:
            x_n = layer.norm1(x)
            _, w = layer.self_attn(x_n, x_n, x_n,
                                   need_weights=True, average_attn_weights=True)
            maps.append(w.cpu())
            x = x + layer.dropout1(
                layer.self_attn(x_n, x_n, x_n, need_weights=False)[0]
            )
            x_n2 = layer.norm2(x)
            x = x + layer.dropout2(layer.linear2(
                layer.dropout(layer.activation(layer.linear1(x_n2)))
            ))
        return maps


it_model = iTransformer(SEQ_LEN, N_FEATURES, d_model=128, nhead=4, n_layers=2)
it_model.load_state_dict(
    torch.load(MODEL_DIR / 'itransformer_best.pt', map_location='cpu')
)
it_model = it_model.to(DEVICE).eval()
print(f'iTransformer 로드 완료 | 파라미터: {sum(p.numel() for p in it_model.parameters()):,}')

## 4. 클러스터별 시계열 추출

각 클러스터의 대표 종목들에서 60일 시퀀스를 추출하고
backbone 학습과 동일한 **instance normalization** (피처별 Z-score)을 적용합니다.

In [ ]:
%%time

parquet_path = DATA_DIR / 'kospi_valid.parquet'
assert parquet_path.exists(), f'parquet 없음: {parquet_path}'

df_all = pd.read_parquet(parquet_path)

# 구조 진단
print(f'shape    : {df_all.shape}')
print(f'index    : {type(df_all.index).__name__} | 샘플: {df_all.index[:2].tolist()}')
print(f'columns  : {df_all.columns.tolist()[:10]}')
print(f'종목코드  : {"있음" if "종목코드" in df_all.columns else "없음"}')
missing_feats = [c for c in FEATURE_COLS if c not in df_all.columns]
print(f'피처 누락 : {missing_feats if missing_feats else "없음 (전부 있음)"}')

# 날짜 컬럼 추출
if hasattr(df_all.index, 'dtype') and 'datetime' in str(df_all.index.dtype):
    df_all = df_all.reset_index()
    date_col = df_all.columns[0]
elif df_all.index.dtype == object:
    try:
        df_all.index = pd.to_datetime(df_all.index)
        df_all = df_all.reset_index()
        date_col = df_all.columns[0]
    except Exception:
        date_col = None
else:
    date_col = next((c for c in df_all.columns
                     if 'date' in c.lower() or '날짜' in c), None)

if date_col:
    df_all[date_col] = pd.to_datetime(df_all[date_col])
    print(f'date_col : {date_col!r} | 범위: {df_all[date_col].min().date()} ~ {df_all[date_col].max().date()}')
else:
    print('[경고] 날짜 컬럼 없음 → 전체 기간 사용')
print(f'종목 수  : {df_all["종목코드"].nunique()}')

FEATURE_COLS_USED = [c for c in FEATURE_COLS if c in df_all.columns]
print(f'사용 피처: {len(FEATURE_COLS_USED)}/{len(FEATURE_COLS)}')


def instance_normalize(seqs):
    mean = seqs.mean(axis=1, keepdims=True)
    std  = seqs.std(axis=1, keepdims=True) + 1e-8
    return (seqs - mean) / std


def extract_cluster_seqs(df_all, cluster_codes, feature_cols,
                          seq_len=60, n_stocks=30, stride=10,
                          test_start='2025-01-01', date_col=None):
    if date_col:
        ts = pd.Timestamp(test_start)
        df_filt = df_all[df_all[date_col] >= ts].copy()
        if len(df_filt) < len(df_all) * 0.05:
            print(f'  [fallback] test 기간 {len(df_filt)}행 → 전체 사용')
            df_filt = df_all.copy()
    else:
        df_filt = df_all.copy()

    # 종목코드 타입 통일: '000020' → 20, 20 → 20 으로 int 변환 후 비교
    df_filt['_code_int'] = df_filt['종목코드'].apply(
        lambda x: int(str(x).lstrip('0') or '0')
    )
    valid_int = set(df_filt['_code_int'].unique())
    codes = [int(c) for c in cluster_codes if int(c) in valid_int][:n_stocks]

    all_seq = []
    for code in codes:
        stock = (df_filt[df_filt['_code_int'] == code]
                 .sort_values(date_col if date_col else df_filt.columns[0])
                 .reset_index(drop=True))
        if len(stock) < seq_len:
            continue
        vals = stock[feature_cols].values.astype(np.float32)
        for end in range(len(vals), seq_len - 1, -stride):
            seg = vals[end - seq_len:end]
            if not np.isnan(seg).any():
                all_seq.append(seg)

    return instance_normalize(np.stack(all_seq)) if all_seq else None


cluster_seqs = {}
for c in sorted(cluster_df['cluster'].unique()):
    codes = cluster_df[cluster_df['cluster'] == c]['종목코드'].tolist()
    seqs  = extract_cluster_seqs(df_all, codes, FEATURE_COLS_USED, date_col=date_col)
    if seqs is not None:
        cluster_seqs[c] = seqs
        print(f'{CLUSTER_LABELS[c]:30s} -> {len(seqs):4d} 시퀀스')
    else:
        print(f'{CLUSTER_LABELS[c]:30s} -> 시퀀스 없음')

print(f'\n총 {len(cluster_seqs)}개 클러스터 준비됨')
assert len(cluster_seqs) > 0, '실패. 위 출력을 공유해주세요.'

## 5. 클러스터별 어텐션 맵 추출

In [ ]:
def get_cluster_attn(model, seqs, batch_size=256, device=DEVICE):
    model.eval()
    n_layers  = len(model.transformer.layers)
    attn_sums = [np.zeros((N_FEATURES, N_FEATURES)) for _ in range(n_layers)]
    n_total   = 0
    for start in range(0, len(seqs), batch_size):
        batch = torch.from_numpy(seqs[start:start + batch_size]).to(device)
        maps  = model.get_attn_maps(batch)
        for l, w in enumerate(maps):
            attn_sums[l] += w.sum(dim=0).numpy()
        n_total += len(batch)
    return [s / n_total for s in attn_sums]


cluster_attns = {}

for c, seqs in cluster_seqs.items():
    cluster_attns[c] = get_cluster_attn(it_model, seqs)
    print(f'{CLUSTER_LABELS[c]:30s} -> 완료 ({len(seqs)}개 시퀀스)')

print('완료')

## 6. 클러스터 간 어텐션 패턴 비교

마지막 레이어 어텐션 맵 기준. 밝은 셀일수록 해당 피처 쌍이 강하게 상호참조됩니다.

In [ ]:
assert len(cluster_attns) > 0, \
    "cluster_attns가 비어있습니다. 셀[10](시퀀스 추출) → 셀[12](어텐션 추출) 순서로 먼저 실행하세요."

N_CLUSTERS = len(cluster_attns)
LAST_LAYER = -1

fig, axes = plt.subplots(1, N_CLUSTERS, figsize=(5 * N_CLUSTERS, 5.5))
if N_CLUSTERS == 1:
    axes = [axes]

all_vals = np.concatenate([cluster_attns[c][LAST_LAYER].flatten()
                            for c in sorted(cluster_attns)])
vmax = np.percentile(all_vals, 97)

for ax, c in zip(axes, sorted(cluster_attns)):
    attn = cluster_attns[c][LAST_LAYER]
    im   = ax.imshow(attn, cmap='YlOrRd', vmin=0, vmax=vmax, aspect='auto')
    ax.set_xticks(range(N_FEATURES))
    ax.set_yticks(range(N_FEATURES))
    ax.set_xticklabels(FEATURE_COLS, rotation=90, fontsize=6.5)
    ax.set_yticklabels(FEATURE_COLS, fontsize=6.5)
    ax.set_title(CLUSTER_LABELS[c], fontsize=9.5,
                 color=CLUSTER_COLORS[c], fontweight='bold')

fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(cmap='YlOrRd',
                             norm=mcolors.Normalize(vmin=0, vmax=vmax))
fig.colorbar(sm, cax=cbar_ax, label='Attention Weight')

fig.suptitle('DTW 클러스터별 iTransformer Variate-Attention (마지막 레이어)',
             fontsize=13, y=1.01)
plt.savefig(RESULT_DIR / 'cluster_attn_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: cluster_attn_heatmap.png')

## 7. 피처 중요도 비교

각 피처가 **받는** 평균 어텐션 (column-mean)을 클러스터별로 비교합니다.

In [ ]:
importance = {}
for c in sorted(cluster_attns):
    attn = cluster_attns[c][LAST_LAYER]
    importance[c] = attn.mean(axis=0)

imp_df = pd.DataFrame(importance, index=FEATURE_COLS)
imp_df.columns = [CLUSTER_LABELS[c] for c in sorted(cluster_attns)]

print('=== 클러스터별 상위 5 피처 ===')
for col in imp_df.columns:
    top5 = imp_df[col].nlargest(5).index.tolist()
    print(f'{col}: {top5}')

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(N_FEATURES)
w = 0.15
for i, c in enumerate(sorted(cluster_attns)):
    ax.bar(x + i * w, imp_df.iloc[:, i], width=w,
           label=CLUSTER_LABELS[c], color=CLUSTER_COLORS[c],
           alpha=0.85, edgecolor='k', lw=0.3)

ax.set_xticks(x + w * 2)
ax.set_xticklabels(FEATURE_COLS, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Average Attention Received')
ax.set_title('피처별 어텐션 중요도 — 클러스터 비교')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig(RESULT_DIR / 'cluster_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 클러스터 특성 해석 — 강한 어텐션 피처 쌍

In [ ]:
def top_pairs(attn_map, feature_cols, top_k=5, exclude_self=True):
    N = len(feature_cols)
    pairs = []
    for i in range(N):
        for j in range(N):
            if exclude_self and i == j:
                continue
            pairs.append((attn_map[i, j], feature_cols[i], feature_cols[j]))
    pairs.sort(reverse=True)
    return pairs[:top_k]


print('=== 클러스터별 강한 어텐션 피처 쌍 (Query -> Key) ===')
for c in sorted(cluster_attns):
    attn  = cluster_attns[c][LAST_LAYER]
    pairs = top_pairs(attn, FEATURE_COLS)
    print(f'\n{CLUSTER_LABELS[c]}')
    for score, q, k in pairs:
        print(f'  {q:20s} -> {k:20s}  ({score:.4f})')

## 9. 레이어 깊이별 어텐션 변화

논문: '얕은 레이어는 현재 상관관계, 깊은 레이어는 미래 패턴을 표현'
가장 큰 클러스터(C3 혼합 대형)로 확인합니다.

In [ ]:
REF_CLUSTER = max(cluster_attns, key=lambda c: len(cluster_seqs[c]))
n_layers    = len(cluster_attns[REF_CLUSTER])

fig, axes = plt.subplots(1, n_layers, figsize=(6 * n_layers, 5.5))
if n_layers == 1:
    axes = [axes]

for l, ax in enumerate(axes):
    attn = cluster_attns[REF_CLUSTER][l]
    im   = ax.imshow(attn, cmap='Blues', vmin=0, aspect='auto')
    ax.set_xticks(range(N_FEATURES))
    ax.set_yticks(range(N_FEATURES))
    ax.set_xticklabels(FEATURE_COLS, rotation=90, fontsize=6.5)
    ax.set_yticklabels(FEATURE_COLS, fontsize=6.5)
    ax.set_title(f'Layer {l + 1}', fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046)

fig.suptitle(f'레이어별 어텐션 변화 ({CLUSTER_LABELS[REF_CLUSTER]})', fontsize=12)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'layer_depth_attn.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. 결과 저장

In [ ]:
imp_df.to_csv(RESULT_DIR / 'cluster_feature_importance.csv', encoding='utf-8-sig')

for c in cluster_attns:
    arr = np.stack(cluster_attns[c])
    np.save(RESULT_DIR / f'attn_cluster{c}.npy', arr)

print('저장 완료:')
for fname in ['cluster_attn_heatmap.png', 'cluster_feature_importance.png',
              'layer_depth_attn.png', 'cluster_feature_importance.csv']:
    print(f'  {RESULT_DIR}/{fname}')
print()
print('활용 방안:')
print('  - 졸업논문 interpretability 섹션: cluster_attn_heatmap.png')
print('  - 9월 뉴스 Cross-Attention 설계 시 클러스터별 지배 피처 참고')